# Phase 1: split + YOLOX ROI extraction (winner pipeline)
Kaggle, Internet ON, GPU ON. Inputs to attach: the RSNA competition + the dataset `dangnh0611/rsna-breast-cancer-detection-best-ckpts`.

In [ ]:
import os
if not os.path.isdir("/kaggle/working/mammogram-cancer-classifier"):
    !git clone https://github.com/hassenmohamed-mezr/mammogram-cancer-classifier.git /kaggle/working/mammogram-cancer-classifier
%cd /kaggle/working/mammogram-cancer-classifier
!git pull
!pip -q install pydicom pylibjpeg pylibjpeg-openjpeg pylibjpeg-libjpeg
# if DICOM decoding fails later:  !pip -q install python-gdcm

### Add YOLOX (winner's breast detector)
1. Right sidebar -> **Add Input** -> search `rsna-breast-cancer-detection-best-ckpts` (by dangnh0611) -> add.
2. Run the next cell: installs YOLOX's small dependencies, clones the winner's repo (code only, ~80 MB, MIT licence) and finds the checkpoint.

In [ ]:
!pip -q install loguru thop pycocotools
WINNER = "/kaggle/working/rsna_winner"
if not os.path.isdir(WINNER):
    !git clone --depth 1 -b reproduce https://github.com/dangnh0611/kaggle_rsna_breast_cancer.git {WINNER}

import glob
hits = glob.glob("/kaggle/input/**/yolox_nano_416_roi_torch.pth", recursive=True)
print("checkpoint:", hits)
assert hits, "Add the best-ckpts dataset as an input (see step 1)"
CKPT = hits[0]

In [ ]:
!ls /kaggle/input
!ls /kaggle/input/rsna-breast-cancer-detection

In [ ]:
# ---- EDIT only if the ls above shows different names ----
COMP     = "/kaggle/input/rsna-breast-cancer-detection"
CSV      = f"{COMP}/train.csv"
IMG_ROOT = f"{COMP}/train_images"
IMG_TMPL = "{root}/{patient_id}/{image_id}.dcm"
OUT      = "/kaggle/working/splits"

In [ ]:
# eval-neg-ratio 10: caps negatives per positive in val/test to save ROI time
# (AUC unaffected; PR-AUC is inflated -> mention it in the results)
!python -m src.data --csv {CSV} --img-root {IMG_ROOT} --img-template "{IMG_TMPL}" --out {OUT} --ratios 1 3 --eval-neg-ratio 10

In [ ]:
import pandas as pd
tr = pd.read_csv(f"{OUT}/train_3to1.csv"); va = pd.read_csv(f"{OUT}/val.csv"); te = pd.read_csv(f"{OUT}/test.csv")
assert not (set(tr.patient_id) & set(va.patient_id)) and not (set(tr.patient_id) & set(te.patient_id)) and not (set(va.patient_id) & set(te.patient_id))
sample = tr.sample(10, random_state=0)
missing = [p for p in sample.path if not os.path.exists(p)]
print("missing paths:", missing[:3], "->", "OK" if not missing else "FIX IMG_TMPL / IMG_ROOT")

In [ ]:
# build the YOLOX detector (loads the winner's weights)
from src.dicom_roi import WinnerRoi, load_dicom, process
roi = WinnerRoi(WINNER, CKPT)          # uses the GPU if available
print("device:", roi.device)

In [ ]:
import numpy as np, matplotlib.pyplot as plt

rows = pd.concat([tr[tr.cancer == 1].sample(10, random_state=1), tr[tr.cancer == 0].sample(10, random_state=1)])
fig, axes = plt.subplots(4, 5, figsize=(18, 14))
for ax, (_, r) in zip(axes.ravel(), rows.iterrows()):
    crop, info = process(load_dicom(r.path), roi)
    conf = "" if info["conf"] is None else f" {info['conf']:.2f}"
    ax.imshow(crop, cmap="gray"); ax.axis("off")
    ax.set_title(f"cancer={r.cancer} | {info['source']}{conf} | {crop.shape[1]}x{crop.shape[0]}", fontsize=9)
plt.tight_layout(); plt.show()

**Check the grid:** most titles should say `yolo`, the whole breast must be inside each crop, no inverted images.
If many say `otsu`/`full`, or boxes are wrong, stop and send me the grid.

## Batch ROI on the whole manifest
Dry run first (50 images). The run is **resumable**: if the session dies, rerun the same cell and it continues.

In [ ]:
from src.run_roi import run
ROI_OUT = "/kaggle/working/roi_out"
run(f"{OUT}/roi_manifest.csv", ROI_OUT, roi=roi, limit=50)
r = pd.read_csv(f"{ROI_OUT}/roi_results.csv")
print(r.source.value_counts())

In [ ]:
# FULL RUN (~7k images). Prints progress + ETA every 200 images.
run(f"{OUT}/roi_manifest.csv", ROI_OUT, roi=roi)

In [ ]:
r = pd.read_csv(f"{ROI_OUT}/roi_results.csv")
print(len(r), "images |", r.source.value_counts().to_dict())
print("errors file:", os.path.exists(f"{ROI_OUT}/roi_errors.csv"))

**After the full run:** save `/kaggle/working/roi_out` and `/kaggle/working/splits` as a Kaggle Dataset
(notebook -> Output -> "New Dataset"). `/kaggle/working` is wiped when the session ends.
The training notebook reads from that dataset.